# 00 — Colab Setup

Run **all cells in order** at the start of every Colab session. Idempotent — re-running won't break anything.

**Before first run:** in Colab, go to *Tools → Secrets* and add a secret named `HF_TOKEN` with your HuggingFace token, and a secret named `WANDB_API_KEY` if you want W&B logging. Toggle the *Notebook access* switch on for both.

## 1. Mount Drive

Drive is the only persistent storage across Colab sessions. Everything else gets wiped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/memory-smolvla'
for sub in ['hf_cache', 'checkpoints', 'results', 'wandb_state']:
    os.makedirs(f'{PROJECT_ROOT}/{sub}', exist_ok=True)
print('Project root:', PROJECT_ROOT)
!df -h /content/drive/MyDrive | tail -1

## 2. Symlink HF cache to Drive

**Critical.** Without this symlink, every Colab session re-downloads the LIBERO dataset (~30GB). With it, you download once and reuse forever.

In [ ]:
import os, shutil
HF_HOME = os.path.expanduser('~/.cache/huggingface')
DRIVE_HF = f'{PROJECT_ROOT}/hf_cache'

if os.path.islink(HF_HOME):
    print(f'Already symlinked: {HF_HOME} -> {os.readlink(HF_HOME)}')
elif os.path.exists(HF_HOME):
    print(f'Removing existing {HF_HOME} (was a regular dir, not a symlink)')
    shutil.rmtree(HF_HOME)
    os.symlink(DRIVE_HF, HF_HOME)
    print(f'Symlinked {HF_HOME} -> {DRIVE_HF}')
else:
    os.makedirs(os.path.dirname(HF_HOME), exist_ok=True)
    os.symlink(DRIVE_HF, HF_HOME)
    print(f'Symlinked {HF_HOME} -> {DRIVE_HF}')

## 3. Clone repo & checkout the merged branch

Replace `<your-fork>` with your fork URL or use the original repo if you have push access.

In [ ]:
REPO_URL = 'https://github.com/<your-fork>/memory-smolVLA.git'  # <-- EDIT ME
BRANCH = 'claude/feature/v5-all-fixes'
REPO_DIR = '/content/memory-smolVLA'

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    !cd {REPO_DIR} && git checkout {BRANCH}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull --ff-only

%cd {REPO_DIR}
!git log --oneline -3

## 4. Install dependencies

Pinned versions for reproducibility. Skip the editable install if it's already done.

In [ ]:
!apt-get install -qq -y libosmesa6 ffmpeg

# LIBERO sim deps (pinned to avoid surprise breaks)
!pip install -q robosuite==1.4.1 mujoco==3.6.0 scipy

# Project (editable). Includes lerobot[smolvla]
!pip install -q -e "{REPO_DIR}[dev]"

# LIBERO benchmark suite (separate repo)
!pip install -q git+https://github.com/Lifelong-Robot-Learning/LIBERO.git

# wandb if you'll use it
!pip install -q wandb

# Lock for next session reproducibility
!pip freeze > {PROJECT_ROOT}/requirements_colab.lock.txt
print(f'Lock file: {PROJECT_ROOT}/requirements_colab.lock.txt')

## 5. HuggingFace + W&B login

Secrets come from the Colab UI (left sidebar key icon). If you haven't set them, this cell will print a warning and you'll need to.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    !huggingface-cli login --token $HF_TOKEN --add-to-git-credential
    print('HF login OK')
except Exception as e:
    print(f'HF_TOKEN not set in Colab Secrets: {e}')
    print('Add it via the key icon in the left sidebar.')

try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    !wandb login --relogin
    print('wandb login OK')
except Exception as e:
    print(f'WANDB_API_KEY not set: {e}')
    print('OK to skip if you do not want wandb logging — set wandb_project: null in your config.')

## 6. Set MUJOCO_GL and verify GPU

MUJOCO_GL=osmesa is required for headless LIBERO sim on Colab. Verify here so the train cells don't fail later.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

import torch
print(f'\ntorch={torch.__version__}, CUDA available={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

## 7. Smoke test (100 training steps, ~3 min)

Run this **once per session** before launching a real training run. It catches:
* config typos
* missing tokens
* dataset access errors
* OOM at this batch / window size
* Drive write permission issues

If it completes, the env is ready for full training.

In [ ]:
%cd {REPO_DIR}
!python scripts/train.py \
    --config configs/libero_v5_run0_diagnostic.yaml \
    --steps 100

If the smoke test passes, you're ready. Open `01_train.ipynb` to launch the real run.